# Per-language GSM8K LoRA fine-tune with before/after evaluation

Same as `train_gsm8k_per_language.ipynb`, plus it measures **train loss and test loss before and after training**.
Set `GSM_LANG`, then **restart the kernel and run all cells once**.

The "before" numbers are the untouched base model (the LoRA adapter starts at zero, so it changes nothing until training begins).
Train loss is measured on a fixed random sample of the training set in eval mode (no dropout), so it is directly comparable to test loss.

In [1]:
import os

# ---------------- Settings ----------------
GSM_LANG   = os.environ.get("GSM_LANG", "mix-full")   # change this per run (or set the GSM_LANG env var)

BASE_MODEL = "google/gemma-3-12b-it"
DATA_ROOT  = "../datasets"                      # relative to this notebook (fine_tuning/)
OUT_DIR    = f"gemma3-12b-gsm8k-{GSM_LANG}"

COMPLETION_ONLY     = True   # True = loss only on the answer tokens (not the question)
MAX_LENGTH          = 1024
TRAIN_EVAL_EXAMPLES = 500    # train examples used for the train-loss measurement; None = whole train set (slow)
TEST_EVAL_EXAMPLES  = None   # None = whole test set
EVAL_STEPS          = None   # e.g. 200 to also evaluate during training; None = only before and after
SEED                = 42

# HF_CACHE = "/path/to/big/storage/hf_cache"  # uncomment if home quota is small
# os.environ["HF_HOME"] = HF_CACHE            # must run before importing transformers
print("Language:", GSM_LANG, "| output:", OUT_DIR)

Language: mix | output: gemma3-12b-gsm8k-mix


## 1. Load the language's train and test sets

In [2]:
from datasets import load_dataset

folder = os.path.join(DATA_ROOT, f"gsm8k-{GSM_LANG}")
train_path = os.path.join(folder, "train.jsonl")
test_path  = os.path.join(folder, "test.jsonl")

for p in (train_path, test_path):
    if not os.path.exists(p):
        raise FileNotFoundError(f"{p} not found. Files in {folder}: {os.listdir(folder) if os.path.isdir(folder) else 'FOLDER MISSING'}")

def prep(ex):
    q = ex["question"].strip()
    a = ex["answer"].strip()
    if COMPLETION_ONLY:
        return {"prompt": [{"role": "user", "content": q}],
                "completion": [{"role": "assistant", "content": a}]}
    return {"messages": [{"role": "user", "content": q},
                         {"role": "assistant", "content": a}]}

def load_split(path):
    d = load_dataset("json", data_files=path, split="train")
    missing = {"question", "answer"} - set(d.column_names)
    if missing:
        raise ValueError(f"{path}: missing columns {missing}; found {d.column_names}")
    n = len(d)
    d = d.filter(lambda ex: bool(ex["question"] and ex["question"].strip() and ex["answer"] and ex["answer"].strip()))
    print(f"{os.path.basename(path)}: {n} rows, {len(d)} usable")
    return d.map(prep, remove_columns=d.column_names)

def subsample(d, n):
    # Fixed random subset (same seed every time, so before/after use identical examples)
    if n is None or len(d) <= n:
        return d
    return d.shuffle(seed=SEED).select(range(n))

train_ds = load_split(train_path).shuffle(seed=SEED)
test_ds  = load_split(test_path)

eval_sets = {
    "train": subsample(train_ds, TRAIN_EVAL_EXAMPLES),
    "test":  subsample(test_ds,  TEST_EVAL_EXAMPLES),
}

print("Train (for training):", len(train_ds))
print("Eval sets:", {k: len(v) for k, v in eval_sets.items()})
print("Optimizer steps for 1 epoch (batch 1 x accum 16):", len(train_ds) // 16)

Generating train split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/255 [00:00<?, ? examples/s]

train.jsonl: 255 rows, 255 usable


Map:   0%|          | 0/255 [00:00<?, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/45 [00:00<?, ? examples/s]

test.jsonl: 45 rows, 45 usable


Map:   0%|          | 0/45 [00:00<?, ? examples/s]

Train (for training): 255
Eval sets: {'train': 255, 'test': 45}
Optimizer steps for 1 epoch (batch 1 x accum 16): 15


## 2. Tokenizer, template check, and length check

In [3]:
from transformers import AutoTokenizer
import numpy as np

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def as_messages(ex):
    return ex["prompt"] + ex["completion"] if COMPLETION_ONLY else ex["messages"]

sample = train_ds.select(range(min(3000, len(train_ds))))
lengths = []
for i, ex in enumerate(sample):
    try:
        ids = tokenizer.apply_chat_template(as_messages(ex), tokenize=True, return_dict=True)["input_ids"]
    except Exception as e:
        print("Bad example:", i, e); break
    lengths.append(len(ids))

lengths = np.array(lengths)
print(f"Checked {len(lengths)} examples | median {int(np.median(lengths))} | p95 {int(np.percentile(lengths, 95))} | max {lengths.max()}")
print(f"Over MAX_LENGTH ({MAX_LENGTH}): {(lengths > MAX_LENGTH).mean():.1%}")

Checked 255 examples | median 197 | p95 385 | max 617
Over MAX_LENGTH (1024): 0.0%


## 3. Load the model in 4-bit

In [4]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)
print("Model loaded")

Loading weights:   0%|          | 0/1065 [00:00<?, ?it/s]

Model loaded


## 4. Build the trainer and evaluate BEFORE training

In [5]:
import json, math
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

args = SFTConfig(
    output_dir=OUT_DIR,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    num_train_epochs=1,
    lr_scheduler_type="cosine",
    logging_steps=10,
    eval_strategy="steps" if EVAL_STEPS else "no",
    eval_steps=EVAL_STEPS,
    save_steps=200,
    save_total_limit=2,
    bf16=True,
    optim="paged_adamw_8bit",
    max_length=MAX_LENGTH,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_sets,          # dict -> metrics are named eval_train_* and eval_test_*
    processing_class=tokenizer,
    peft_config=peft_config,
)

def run_eval(label):
    raw = trainer.evaluate()
    out = {}
    for name in eval_sets:
        loss = raw[f"eval_{name}_loss"]
        out[name] = {"loss": loss, "perplexity": math.exp(loss)}
        acc = raw.get(f"eval_{name}_mean_token_accuracy")
        if acc is not None:
            out[name]["token_accuracy"] = acc
    print(f"\n=== {label} ===")
    for name, m in out.items():
        extra = f" | token acc {m['token_accuracy']:.4f}" if "token_accuracy" in m else ""
        print(f"{name:5s} loss {m['loss']:.4f} | perplexity {m['perplexity']:.3f}{extra}")
    return out

before = run_eval("BEFORE training (base model)")

Tokenizing train dataset:   0%|          | 0/255 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/255 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/255 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/255 [00:00<?, ? examples/s]

Tokenizing test dataset:   0%|          | 0/45 [00:00<?, ? examples/s]

Building labels for test dataset:   0%|          | 0/45 [00:00<?, ? examples/s]

Truncating test dataset:   0%|          | 0/45 [00:00<?, ? examples/s]

Dropping fully masked examples from test dataset:   0%|          | 0/45 [00:00<?, ? examples/s]

Training Loss,Validation Loss,Step,Train Loss,Entropy,Num Tokens,Mean Token Accuracy
No log,No log,0,1.402496,0.355456,0.000000,0.770530


Training Loss,Validation Loss,Step,Test Loss,Entropy,Num Tokens,Mean Token Accuracy
No log,No log,0,1.892489,0.419432,0.000000,0.701075



=== BEFORE training (base model) ===
train loss 1.4025 | perplexity 4.065
test  loss 1.8925 | perplexity 6.636


## 5. Train and save

In [6]:
trainer.train()

# Save first, so a failure in the evaluation below can never cost you the adapter
FINAL_DIR = os.path.join(OUT_DIR, "final")
trainer.save_model(FINAL_DIR)            # adapter_config.json + adapter_model.safetensors
tokenizer.save_pretrained(FINAL_DIR)
print("Saved adapter to", os.path.abspath(FINAL_DIR))

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2}.


Step,Training Loss
10,0.873949


Saved adapter to /workspace/NLP-and-LLMs-Math-by-Language-Project/fine_tuning/gemma3-12b-gsm8k-mix/final


## 6. Evaluate AFTER training and compare

In [7]:
after = run_eval("AFTER training")

print("\n=== Change (after - before) ===")
for name in eval_sets:
    d = after[name]["loss"] - before[name]["loss"]
    print(f"{name:5s} loss {before[name]['loss']:.4f} -> {after[name]['loss']:.4f}  (delta {d:+.4f})")

gap = after["test"]["loss"] - after["train"]["loss"]
print(f"\nTest minus train loss after training: {gap:+.4f}")

results = {
    "language": GSM_LANG,
    "n_train_for_training": len(train_ds),
    "n_eval": {k: len(v) for k, v in eval_sets.items()},
    "before": before,
    "after": after,
}
with open(os.path.join(OUT_DIR, "eval_results.json"), "w") as f:
    json.dump(results, f, indent=2)
print("Wrote", os.path.join(OUT_DIR, "eval_results.json"))

Training Loss,Validation Loss,Step,Train Loss,Entropy,Num Tokens,Mean Token Accuracy
0.873949,No log,16,0.543857,0.550582,55032.000000,0.856724


Training Loss,Validation Loss,Step,Test Loss,Entropy,Num Tokens,Mean Token Accuracy
0.873949,No log,16,0.811856,0.698440,55032.000000,0.803819



=== AFTER training ===
train loss 0.5439 | perplexity 1.723
test  loss 0.8119 | perplexity 2.252

=== Change (after - before) ===
train loss 1.4025 -> 0.5439  (delta -0.8586)
test  loss 1.8925 -> 0.8119  (delta -1.0806)

Test minus train loss after training: +0.2680
Wrote gemma3-12b-gsm8k-mix/eval_results.json


## Notes
- **Reading the results:** a healthy run has both losses dropping from "before" to "after", with test loss a bit above train loss. If train loss falls a lot while test loss stays flat or rises, the adapter is overfitting.
- **Cost:** each evaluation passes the 500 train examples plus the whole test set through the model one at a time, so expect a few minutes for each of the before and after checks. Lower `TEST_EVAL_EXAMPLES` or `TRAIN_EVAL_EXAMPLES` to speed it up.
- **Interrupted run:** restart the kernel, re-run sections 0 to 4 (you can skip the "before" evaluation by commenting out its last line), then change `trainer.train()` to `trainer.train(resume_from_checkpoint=True)`.
- **Use the adapter:** set `ADAPTER_PATH = "gemma3-12b-gsm8k-<lang>/final"` in `run_finetuned.ipynb`.